# Is It Necessary to Use a Specialized In-Memory Database?

A Python launcher invokes the real Apache-licensed esProc Java engine, not a Python implementation of SPL. Requires JDK 17 plus a Python kernel. The example counts consecutive strictly positive changes between observed quotes (four rising prices give three changes), not missing calendar days. Synthetic fixtures test correctness, not database throughput or trading performance.

Run cells in order. Dependencies and hardware requirements are listed below.

In [ ]:
import os, shutil, hashlib, tempfile, subprocess, itertools, random, math
from pathlib import Path
from urllib.request import urlopen
# Install a JDK 17 (for example Eclipse Temurin). Set TDB_JAVA_HOME if java is not on PATH.
java_home = os.environ.get('TDB_JAVA_HOME')
java = str(Path(java_home)/'bin/java') if java_home else shutil.which('java')
if not java:
    raise RuntimeError('Install JDK 17 and set TDB_JAVA_HOME to its home directory.')
print(subprocess.run([java, '-version'], capture_output=True, text=True, check=True).stderr.strip())
work = tempfile.TemporaryDirectory(prefix='tdb-spl-')
root = Path(work.name)
jar = root/'esproc-20260507.jar'
with urlopen('https://repo.maven.apache.org/maven2/com/scudata/esproc/esproc/20260507/esproc-20260507.jar', timeout=60) as response:
    data = response.read()
assert hashlib.sha256(data).hexdigest() == 'cb16ee379f40254cd12a727c3107c1857da369ff4f1385d973af8b5d2f49d0e6'
jar.write_bytes(data)
print('Verified real esProc engine: 20260507')

In [ ]:
JAVA_SOURCE = 'import java.io.*;\nimport com.scudata.dm.Context;\nimport com.scudata.dm.Table;\nimport com.scudata.expression.Expression;\npublic class RisingRuns {\n public static void main(String[] args) throws Exception {\n  BufferedReader in = new BufferedReader(new InputStreamReader(System.in));\n  String line;\n  while ((line=in.readLine())!=null) {\n   Table stock=new Table(new String[]{"date","price"});\n   if (!line.isEmpty()) for (String record:line.split(";")) {\n    String[] fields=record.split(":");\n    stock.newLast(new Object[]{Integer.valueOf(fields[0]),Double.valueOf(fields[1])});\n   }\n   Context ctx=new Context();ctx.setParamValue("stock",stock);\n   if (stock.length()==0) System.out.println(0);\n   else System.out.println(new Expression(ctx,\n      "stock.sort(date).group@i(price<=price[-1]).max(~.len()-1)").calculate(ctx));\n  }\n }\n}'
source = root/'RisingRuns.java'
source.write_text(JAVA_SOURCE, encoding='utf-8')
def spl_runs(cases):
    for records in cases:
        dates = [date for date,price in records]
        if any(type(date) is not int for date in dates) or len(set(dates)) != len(dates):
            raise ValueError('Use unique integer date ordinals.')
        if any(not isinstance(price, (int,float)) or isinstance(price,bool) or not math.isfinite(price)
               for date,price in records):
            raise ValueError('Prices must be finite numeric values.')
    payload = ''.join(';'.join(f'{date}:{price}' for date,price in records)+'\n' for records in cases)
    result = subprocess.run([java, '-Djava.awt.headless=true', '--class-path', str(jar), str(source)],
        input=payload, text=True, capture_output=True, check=True, timeout=120)
    if result.stderr:
        print(result.stderr, end='')
    values = [int(value) for value in result.stdout.splitlines()]
    assert len(values) == len(cases)
    return values
records = [(3, 12), (1, 10), (4, 11), (2, 11)]
print('Longest strictly rising run in observed changes:', spl_runs([records])[0])

In [ ]:
def reference(records):
    best = current = 0
    ordered = sorted(records)
    for previous, present in zip(ordered, ordered[1:]):
        current = current + 1 if present[1] > previous[1] else 0
        best = max(best, current)
    return best
cases = []
for length in range(8):
    for prices in itertools.product([-1, 0, 1], repeat=length):
        records = list(enumerate(prices)); records.reverse(); cases.append(records)
rng = random.Random(22)
for _ in range(100):
    records = list(enumerate([rng.uniform(-100, 100) for _ in range(rng.randrange(1,40))]))
    rng.shuffle(records); cases.append(records)
actual = spl_runs(cases)
assert actual == [reference(records) for records in cases]
for records in [[(0,float('nan'))], [(0,float('inf'))], [(1,1),(1,2)], [('bad',1)]]:
    try: spl_runs([records])
    except ValueError: pass
    else: raise AssertionError('Invalid input accepted')
assert spl_runs([[],[(0,7)],list(enumerate([1,2,3,4])),list(enumerate([4,3,2,1])),
    list(enumerate([1,1,1,1])),list(enumerate([1,2,2,3]))]) == [0,0,3,0,0,1]
print('Real SPL/reference comparisons:', len(cases), '; empty, single, ties, unsorted, negative and invalid-input checks passed.')
work.cleanup()

## Original archive appendix — not executable

These verbatim source blocks and transcripts preserve the historical examples. They are not the maintained workflow above.

### Original code 1

```text
stock.sort(date).group@i(price>price[-1]).max(~.len()) 
stock.sort(date).group@i(price>price[-1]).max(~.len())
```